# Multi-Source Visual Search Engine (MSSearch) on 2x NVIDIA T4 GPUs
## Benchmark Execution & Set-Based Attribute Evaluation: Pilot 10 Multi-Image Dataset

> **Objective**: Execute **MSSearch** on **2x NVIDIA T4 GPUs (32GB VRAM total)** in pure `float16` precision across the **Pilot 10 Multi-Source Dataset** (`./datasets/pilot_10_samples.json`). Supports both **`N_5` pool mode** (5 candidate images per query: target images + hard distractors) and **All Pools mode** (`N_5`, `N_10`, `N_20`), evaluating open-ended free-form predictions against ground-truth attribute sets.

### 🛡️ Features & Operational Invariants
- **Linear Budget Scaling ($N \times \text{budget}$)**: Stage 4 SAM3 patch scan caps, latent speculative queries, and retained candidate capacities scale dynamically with pool size $N = |\mathcal{I}_{\text{cand}}|$ (`scale_budgets_by_num_images=True`), matching `debug_mssearch_multisource_stages.ipynb`.
- **Multi-Answer Mode & Consensus Consolidation**: Runs with `multi_answer_mode=True` to evaluate all candidate crops across all $N$ images (`enable_early_consensus_exit=False`), consolidating all unique discovered attributes into unified semicolon-separated evidence predictions.
- **Multi-Source Pool Resolution**: Seamlessly loads multi-image candidate sets from local/Kaggle disk mounts or downloads missing Flickr images with resilient user-agent headers and retries.
- **Configurable Pool Modes**:
  - `POOL_MODE = "N_5"`: Evaluates fixed 5-image candidate sets per sample ($K_{\text{gt}}$ targets + hard distractors).
  - `POOL_MODE = "ALL"`: Evaluates across all pool tiers (`N_5`, `N_10`, `N_20`) sequentially.
- **Fail-Safe Incremental Streaming**: Predictions append immediately to `/kaggle/working/answers/mssearch_pilot10_predictions.jsonl` with `os.fsync()` for 100% crash and timeout resilience.
- **Inline Multi-Label Set Evaluation**: Automatically parses free-form VLM responses and benchmarks multi-label precision, recall, and exact match against ground truth attribute lists (`answer.list` / `answer.split_comma`).
- **2x T4 Asymmetric Distribution**: Qwen2.5-VL vision tower & early layers on `cuda:0`, later layers & SAM3 foundation model on `cuda:1`.

In [ ]:
import os
import sys
import gc
import json
import time
import torch
from PIL import Image

print('=' * 75)
print('SYSTEM TELEMETRY & MULTI-GPU DIAGNOSTICS')
print('=' * 75)
print(f'PyTorch Version : {torch.__version__}')
print(f'CUDA Available  : {torch.cuda.is_available()}')
device_count = torch.cuda.device_count()
print(f'GPU Device Count: {device_count}')

if device_count < 2:
    print('[WARNING] Less than 2 GPUs detected. Operating in single-GPU / CPU mode.')
else:
    for i in range(device_count):
        props = torch.cuda.get_device_properties(i)
        vram_gb = props.total_memory / (1024 ** 3)
        print(f'  [GPU {i}] {props.name} | Compute: {props.major}.{props.minor} | VRAM: {vram_gb:.2f} GB')

has_sdpa = hasattr(torch.nn.functional, 'scaled_dot_product_attention')
print(f'PyTorch SDPA Available: {has_sdpa}')
print('=' * 75)

In [ ]:
# ==============================================================================
# DATASET & MODEL PATHS CONFIGURATION (Pilot 10 Multi-Source Dataset)
# ==============================================================================

# Foundation Model Weights
SAM3_CHECKPOINT_PATH = '/kaggle/input/datasets/thnhtt/vs-sam3/sam3/sam3/sam3.pt'
if not os.path.exists(SAM3_CHECKPOINT_PATH):
    for root, _, files in os.walk('/kaggle/input'):
        if 'sam3.pt' in files:
            SAM3_CHECKPOINT_PATH = os.path.join(root, 'sam3.pt')
            break

QWEN_MODEL_PATH = 'Qwen/Qwen2.5-VL-7B-Instruct'
SIGLIP_MODEL_NAME = 'google/siglip-base-patch16-384'
NLP_MODEL_PATH = 'en_core_web_sm'

# Pilot 10 Dataset Path Detection
PILOT_DATASET_PATHS = [
    '/kaggle/input/datasets/thnhtt/vs-dataset/pilot_10_samples.json',
    '/kaggle/input/pilot_10_samples.json',
    './datasets/pilot_10_samples.json',
    '../datasets/pilot_10_samples.json'
]

PILOT_DATASET_PATH = None
for p in PILOT_DATASET_PATHS:
    if os.path.exists(p):
        PILOT_DATASET_PATH = p
        break

if not PILOT_DATASET_PATH:
    PILOT_DATASET_PATH = './datasets/pilot_10_samples.json'

# Local / Mounted Images Cache Directory
IMAGES_CACHE_DIR = '/kaggle/working/data/pilot_1000_images'
LOCAL_SEARCH_DIRS = [
    '/kaggle/input/datasets/thnhtt/vs-dataset/pilot_1000_images',
    '/kaggle/input/pilot_1000_images',
    '/kaggle/working/data/pilot_1000_images',
    './data/pilot_1000_images'
]
os.makedirs(IMAGES_CACHE_DIR, exist_ok=True)

# Execution Knobs
# POOL_MODE options: 'N_5' (run only N_5 pool) or 'ALL' (run N_5, N_10, N_20 pools)
POOL_MODE = 'N_5'
SMOKE_TEST_MODE = False        # Set True to quickly verify first sample
MAX_SAMPLES = None             # None to run all samples

OUTPUT_DIR = '/kaggle/working/answers'
os.makedirs(OUTPUT_DIR, exist_ok=True)
PREDICTIONS_JSONL = os.path.join(OUTPUT_DIR, f'mssearch_pilot10_predictions_{POOL_MODE.lower()}.jsonl')

print(f'[CONFIG] Pilot Dataset Path : {PILOT_DATASET_PATH} (Exists: {os.path.exists(PILOT_DATASET_PATH)})')
print(f'[CONFIG] SAM3 Weights       : {SAM3_CHECKPOINT_PATH} (Exists: {os.path.exists(SAM3_CHECKPOINT_PATH)})')
print(f'[CONFIG] Pool Mode          : {POOL_MODE}')
print(f'[CONFIG] Streaming Output   : {PREDICTIONS_JSONL}')

In [ ]:
# Clone or update MSSearch repository from GitHub using Kaggle Secret 'MaCVi'
import os
import sys
import subprocess
import importlib
import urllib.request

REPO_DIR = '/kaggle/working/MSSearch'
github_token = None

try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    github_token = user_secrets.get_secret('MaCVi')
    print('[SETUP] Successfully retrieved GitHub token from Kaggle User Secret \'MaCVi\'.')
except Exception as e:
    print(f'[SETUP] Note: UserSecretsClient notice ({e}). Checking environment variables...')
    github_token = os.environ.get('MaCVi') or os.environ.get('GITHUB_TOKEN')

if not os.path.exists(REPO_DIR):
    if github_token:
        print(f'[SETUP] Cloning MSSearch repository using token into {REPO_DIR}...')
        clone_url = f'https://{github_token}@github.com/tothanhdat2006/MSSearch.git'
    else:
        print(f'[SETUP] Attempting unauthenticated clone into {REPO_DIR}...')
        clone_url = 'https://github.com/tothanhdat2006/MSSearch.git'
    ret = subprocess.run(['git', 'clone', clone_url, REPO_DIR], capture_output=True, text=True)
    if ret.returncode == 0:
        print(f'[SETUP] Successfully cloned MSSearch into {REPO_DIR}')
    else:
        safe_err = ret.stderr.replace(github_token, '******') if github_token else ret.stderr
        print(f'[SETUP] Git clone failed: {safe_err}')
else:
    print(f'[SETUP] MSSearch repository already exists at {REPO_DIR}. Pulling latest changes from main...')
    ret = subprocess.run(['git', '-C', REPO_DIR, 'pull', 'origin', 'main'], capture_output=True, text=True)
    if ret.returncode == 0:
        print(f'[SETUP] Repository updated to latest: {ret.stdout.strip()}')
    else:
        safe_err = ret.stderr.replace(github_token, '******') if github_token else ret.stderr
        print(f'[SETUP] Git pull note: {safe_err.strip()}')

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
elif sys.path[0] != REPO_DIR:
    sys.path.remove(REPO_DIR)
    sys.path.insert(0, REPO_DIR)

importlib.invalidate_caches()
for mod_name in list(sys.modules.keys()):
    if mod_name.startswith('mssearch'):
        try:
            importlib.reload(sys.modules[mod_name])
        except Exception:
            pass

bpe_target = os.path.join(REPO_DIR, 'assets', 'bpe_simple_vocab_16e6.txt.gz')
if not os.path.exists(bpe_target):
    os.makedirs(os.path.dirname(bpe_target), exist_ok=True)
    bpe_url = 'https://raw.githubusercontent.com/openai/CLIP/main/clip/bpe_simple_vocab_16e6.txt.gz'
    try:
        urllib.request.urlretrieve(bpe_url, bpe_target)
        print('[SETUP] Downloaded BPE vocab asset.')
    except Exception as e:
        print(f'[SETUP WARNING] BPE download: {e}')

get_ipython().system('pip install -q scikit-image scikit-learn networkx iopath decord ftfy')

from mssearch.config import MSSearchConfig
from mssearch.MSSearch import MultiSourceSearchEngine, get_mssearch_response
from mssearch.models.modeling_qwenvl import ModelQwenVL
from mssearch.models.modeling_sam3 import sam3_inference
from mssearch.models.modeling_siglip import SigLIPScorer

print('[SUCCESS] Successfully imported MSSearch core engine and model adapters!')

In [ ]:
print('=' * 75)
print('INITIALIZING FOUNDATION MODELS & DUAL-GPU ASYMMETRIC DISTRIBUTION')
print('=' * 75)

# 1. Load Qwen2.5-VL in float16 (Sharded across GPU 0 & GPU 1)
print('[1/4] Loading Qwen2.5-VL-7B-Instruct (Pure float16 with SDPA backend)...')
qwen_engine = ModelQwenVL(
    model_path=QWEN_MODEL_PATH,
    torch_dtype=torch.float16,
    patch_scale=1.2
)

# 2. Load SAM3 on cuda:1 (5.2GB allocation alongside Qwen layers 15-27)
sam3_device = 'cuda:1' if torch.cuda.device_count() >= 2 else 'cuda:0'
print(f'[2/4] Loading SAM3 image model on {sam3_device}...')
sam3_engine = sam3_inference(
    model_path=SAM3_CHECKPOINT_PATH,
    device=sam3_device
)

# 3. Load SigLIP-base on cuda:0 (Dense spatial token scorer)
siglip_device = 'cuda:0'
print(f'[3/4] Loading SigLIP Scorer ({SIGLIP_MODEL_NAME}) on {siglip_device}...')
siglip_engine = SigLIPScorer(
    model_name=SIGLIP_MODEL_NAME,
    device=siglip_device,
    torch_dtype=torch.float16
)

# 4. Load SpaCy NLP Parser on CPU
print(f'[4/4] Loading SpaCy NLP parser ({NLP_MODEL_PATH})...')
import spacy
try:
    nlp_parser = spacy.load(NLP_MODEL_PATH)
except Exception:
    get_ipython().system('python -m spacy download en_core_web_sm')
    nlp_parser = spacy.load('en_core_web_sm')

print('-' * 75)
for i in range(torch.cuda.device_count()):
    alloc_gb = torch.cuda.memory_allocated(i) / (1024 ** 3)
    res_gb = torch.cuda.memory_reserved(i) / (1024 ** 3)
    total_gb = torch.cuda.get_device_properties(i).total_memory / (1024 ** 3)
    print(f'  [GPU {i}] Allocated: {alloc_gb:.2f} GB | Reserved: {res_gb:.2f} GB | Total: {total_gb:.2f} GB | Headroom: {total_gb - res_gb:.2f} GB')
print('=' * 75)

In [ ]:
# Configure MSSearch for Multi-Source Visual Search Benchmark (METHOD_SPECIFICATION.md Aligned)
config_pilot = MSSearchConfig(
    # Multi-Source & Multi-Answer Knobs (debug_mssearch_multisource_stages.ipynb Aligned)
    multi_answer_mode=True,               # Deactivate Stage 6 early exit; consolidate all unique answers into semicolon text
    scale_budgets_by_num_images=True,     # Linearly scale Stage 4 scan and latent budgets by N images (N * budget)
    answer_dedup_similarity=0.85,         # Semantic deduplication threshold for multi-source answer clusters

    # Phase 1 & 2: Cascaded Direct Gating
    enable_direct_answer=True,           # Phase 1: Direct Global Answer via logit sufficiency probing
    direct_answer_conf_threshold=0.60,   # Sufficiency threshold c_q in [-1, 1] on root view
    enable_fast_path=True,               # Phase 2: Tool-Based Fast Path via SAM3 full-canvas detection
    fast_path_conf_threshold=0.70,       # Existence confidence threshold for fast path instant exit

    # Stage 1: Parsing, Image-Grounded Anchors & Relational Intent
    enable_anchor_first_ordering=True,   # Prioritize physical host entities ahead of abstract attributes
    enable_image_grounded_anchors=True,  # Discover host objects on low-res canvas when text is anchorless
    image_grounded_anchor_resolution=1024,# Low-res glimpse resolution ceiling for vision fallback
    enable_query_adaptive_dual_scale=True,# Dual-Scale visual prompt [Global with BBox, Local Crop] for relations

    # Stage 3: Entropy-Gated Dynamic Saliency-Plateau Partitioning
    entropy_threshold=0.35,              # Threshold separating Quadrant (< 0.35) vs SAM3-SGAP (>= 0.35)
    min_patch_entropy=0.35,              # Preserves smooth painted targets
    partition_mode='hybrid',
    partition_stopping_mode='saliency_plateau', # Dynamic semantic zoom stopping vs fixed_size
    max_patch_size=512,                  # High-resolution boundary standard (used when fixed_size)
    min_plateau_size=256,                # Scale floor for saliency plateau stopping
    max_plateau_size=1536,               # Physical scale ceiling (GPU safety)
    saliency_plateau_gain_threshold=0.04,# Minimum saliency gain required to descend deeper
    saliency_target_threshold=0.35,      # High confidence target threshold where marginal gain halts zoom
    max_patches_per_image=100,

    # Stage 4: Feature Grouping, Micro-Fovea Zoom & Contextual Gating (Base Budgets scaled by N)
    grouping_sim_threshold=0.80,         # Feature cosine similarity threshold for clustering candidate patches
    max_sam3_eval_patches=16,            # Base SAM3 scan budget (scaled dynamically by N candidate images)
    enable_latent_gating=True,
    latent_confidence_threshold=0.0,
    latent_prompt_mode='question_aware', # Question-aware speculative latent prompt
    max_latent_evals=8,                  # Base latent speculative VLM checks (scaled dynamically by N candidate images)
    enable_existence_gating=True,
    existence_confidence_threshold=0.0,
    max_retained_patches=16,             # Base global candidate pool cap (scaled dynamically by N candidate images)

    # Stage 6: Reading, Formatting & Consensus
    bbox_expansion_factor=1.25,          # Independent 1.25x context expansion for foveated crops
    enable_visual_prompt_bbox=True,
    max_vlm_tokens=128,                  # Response token budget
    enable_early_consensus_exit=False,   # Deactivated for multi-source multi-answer discovery across all candidate images
    prompt_directed_final_answer=True,   # Enforce prompt-directed Final Answer format
    include_options_in_prompt=False,     # Open-ended multi-source attribute discovery (free-form)
    answer_type='attribute_list',             # Open-vocabulary generation mode
    verbose=False                        # Clean high-throughput output
)

print('[CONFIG] MSSearch master configuration initialized with linear N-budget scaling and multi-answer synthesis!')

In [ ]:
# ==============================================================================
# RESILIENT MULTI-SOURCE IMAGE RESOLVER & OPENIMAGES S3 / FLICKR FETCHER
# ==============================================================================
import urllib.request
import time
import os
from PIL import Image

USER_AGENT_HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'image/avif,image/webp,image/apng,image/svg+xml,image/*,*/*;q=0.8'
}

OPENIMAGES_SPLITS = ['train', 'validation', 'test']

def resolve_or_download_image(image_id: str, url: str = None, rel_path: str = None) -> Image.Image:
    """
    Attempts to load image from local directories first.
    If not found, fetches from OpenImages S3 bucket (train/val/test splits).
    Falls back to original Flickr URL if S3 is unavailable.
    """
    target_filename = f'{image_id}.jpg'
    
    # 1. Search local paths
    search_paths = []
    if rel_path:
        search_paths.append(rel_path)
    for base_dir in LOCAL_SEARCH_DIRS:
        search_paths.append(os.path.join(base_dir, target_filename))
        search_paths.append(os.path.join(base_dir, image_id))
    
    for sp in search_paths:
        if os.path.exists(sp):
            try:
                return Image.open(sp).convert('RGB')
            except Exception:
                pass
                
    # 2. Check destination cache path
    cache_path = os.path.join(IMAGES_CACHE_DIR, target_filename)
    if os.path.exists(cache_path):
        try:
            return Image.open(cache_path).convert('RGB')
        except Exception:
            pass

    # 3. Primary Network Source: OpenImages S3 (Fast, permanent, no HTTP 429/404)
    for split in OPENIMAGES_SPLITS:
        s3_url = f'https://open-images-dataset.s3.amazonaws.com/{split}/{image_id}.jpg'
        try:
            req = urllib.request.Request(s3_url, headers=USER_AGENT_HEADERS)
            with urllib.request.urlopen(req, timeout=10) as resp:
                if resp.status == 200:
                    img_data = resp.read()
                    with open(cache_path, 'wb') as f_img:
                        f_img.write(img_data)
                    return Image.open(cache_path).convert('RGB')
        except Exception:
            pass

    # 4. Fallback Network Source: Original Flickr URL
    if url:
        for attempt in range(3):
            try:
                req = urllib.request.Request(url, headers=USER_AGENT_HEADERS)
                with urllib.request.urlopen(req, timeout=12) as resp:
                    if resp.status == 200:
                        img_data = resp.read()
                        with open(cache_path, 'wb') as f_img:
                            f_img.write(img_data)
                        return Image.open(cache_path).convert('RGB')
            except Exception as e:
                time.sleep(1.0 * (attempt + 1))

    raise FileNotFoundError(
        f"Unable to retrieve image '{image_id}'. Searched local dirs {LOCAL_SEARCH_DIRS}, "
        f"OpenImages S3 splits {OPENIMAGES_SPLITS}, and original URL '{url}'."
    )

print('[SETUP] Multi-source image resolver ready with OpenImages S3 primary repository and local path caching.')

In [ ]:
import copy
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# Helper function to visualize candidate images with Gold Object BBoxes, Stage 6 crops, and answers
def visualize_sample_results(
    sample_id, pool_name, question, pred_answer, pred_list, gt_list,
    candidate_images, candidate_meta, extractions, stage6_boxes, evidence_images, gt_lookup,
    stage6_crops=None
):
    print("\n" + "=" * 85)
    print(f"[VISUALIZATION] Task: {sample_id}_{pool_name}")
    print(f"Question : {question}")
    print(f"Predicted String: '{pred_answer}'")
    print(f"Predicted List  : {pred_list}")
    print(f"Ground Truth    : {gt_list}")
    print(f"Evidence Image Indices: {evidence_images}")
    print("=" * 85)
    
    # 1. Plot all candidate original images with Ground Truth Gold Bounding Boxes & Evidence Highlight
    num_cand = len(candidate_images)
    cols = min(5, num_cand)
    rows = (num_cand + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4.5 * rows), squeeze=False)
    
    for idx, (img_pil, meta) in enumerate(zip(candidate_images, candidate_meta)):
        r, c = idx // cols, idx % cols
        ax = axes[r][c]
        ax.imshow(img_pil)
        
        is_evidence = idx in evidence_images
        role = meta.get('role', 'unknown')
        iid = meta.get('image_id', '')
        title_color = 'green' if is_evidence else ('darkred' if role == 'hard_distractor' else 'black')
        ax.set_title(
            f"Img {idx} ({role}){' [EVIDENCE]' if is_evidence else ''}\n{iid[:10]}",
            fontsize=9, color=title_color, fontweight='bold' if is_evidence else 'normal'
        )
        ax.axis('off')
        
        # Highlight evidence frame
        if is_evidence:
            border = patches.Rectangle((0, 0), img_pil.width, img_pil.height, linewidth=3, edgecolor='limegreen', facecolor='none')
            ax.add_patch(border)
            
        # Draw Gold Object Ground-Truth Bounding Boxes with Labels
        gt_meta = gt_lookup.get(iid, {})
        gt_bboxes = gt_meta.get('gt_bboxes', [])
        for gb in gt_bboxes:
            # Prefer bbox_pixel [x1, y1, x2, y2], fallback to bbox_norm
            bp = gb.get('bbox_pixel')
            if bp and len(bp) == 4:
                bx1, by1, bx2, by2 = bp
            else:
                bn = gb.get('bbox_norm', [0, 0, 1, 1])
                bx1 = bn[0] * img_pil.width
                by1 = bn[1] * img_pil.height
                bx2 = bn[2] * img_pil.width
                by2 = bn[3] * img_pil.height
                
            bw = max(1.0, bx2 - bx1)
            bh = max(1.0, by2 - by1)
            rect = patches.Rectangle((bx1, by1), bw, bh, linewidth=2, edgecolor='gold', facecolor='none')
            ax.add_patch(rect)
            
            # Format annotation label
            obj_name = gb.get('object', 'Object')
            attrs = gb.get('attributes', {})
            attr_desc = ", ".join(f"{v}" for k, v in attrs.items() if v and k in ['color', 'pattern', 'material', 'shape'])
            label_text = f"{obj_name}: {attr_desc}" if attr_desc else obj_name
            
            ax.text(
                bx1, max(0, by1 - 4), label_text,
                color='black', fontsize=7, fontweight='bold',
                bbox=dict(boxstyle='square,pad=0.2', facecolor='gold', edgecolor='none', alpha=0.85)
            )
            
    # Hide unused subplots
    for idx in range(num_cand, rows * cols):
        axes[idx // cols][idx % cols].axis('off')
    plt.suptitle(f"{sample_id} ({pool_name}) - Candidate Images & Gold Object Ground Truth", fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()
    plt.close(fig)
    
    # 2. Plot Stage 6 Patches / Crops (Two-Tier: Green=Evidence, Red=Filtered)
    display_crops = stage6_crops if stage6_crops else extractions
    if display_crops:
        n_ext = len(display_crops)
        cols_ext = min(5, n_ext)
        rows_ext = (n_ext + cols_ext - 1) // cols_ext
        fig_ext, axes_ext = plt.subplots(rows_ext, cols_ext, figsize=(4.2 * cols_ext, 4.2 * rows_ext), squeeze=False)
        
        # Compute image-level yield diagnostics
        yield_summary = {i: {"ev": 0, "total": 0} for i in range(num_cand)}
        for c in display_crops:
            i_idx = c.get('img_idx', 0)
            if i_idx in yield_summary:
                yield_summary[i_idx]["total"] += 1
                if c.get('is_evidence', True):
                    yield_summary[i_idx]["ev"] += 1
        yield_str = " | ".join([f"Img {k}: {v['ev']}/{v['total']} ev" for k, v in yield_summary.items()])
        print(f"[Stage 6 Yield Diagnostics] {yield_str}")
        
        for e_idx, ext in enumerate(display_crops):
            r, c = e_idx // cols_ext, e_idx % cols_ext
            ax = axes_ext[r][c]
            
            img_idx = ext.get('img_idx', 0)
            base_img = candidate_images[img_idx] if img_idx < len(candidate_images) else candidate_images[0]
            
            # Crop image patch using expanded_box or box
            box = ext.get('expanded_box') or ext.get('box') or [0, 0, base_img.width, base_img.height]
            x1, y1, x2, y2 = [int(round(coord)) for coord in box]
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(base_img.width, x2), min(base_img.height, y2)
            
            if x2 > x1 and y2 > y1:
                crop = base_img.crop((x1, y1, x2, y2))
            else:
                crop = base_img
                
            ax.imshow(crop)
            is_ev = ext.get('is_evidence', True)
            border_color = 'limegreen' if is_ev else 'crimson'
            title_color = 'darkgreen' if is_ev else 'firebrick'
            status_tag = '[EVIDENCE]' if is_ev else '[FILTERED]'
            ans = ext.get('final_answer') or ext.get('raw_answer') or 'None'
            branch = ext.get('source_branch', 'Stage 6')
            ax.set_title(f"Img {img_idx} [{branch}] {status_tag}\nAns: {ans}", fontsize=8, fontweight='bold', color=title_color)
            ax.axis('off')
            
            border = patches.Rectangle((0, 0), crop.width, crop.height, linewidth=3 if is_ev else 2, edgecolor=border_color, facecolor='none', linestyle='-' if is_ev else '--')
            ax.add_patch(border)
            
        for e_idx in range(n_ext, rows_ext * cols_ext):
            axes_ext[e_idx // cols_ext][e_idx % cols_ext].axis('off')
        plt.suptitle(f"{sample_id} ({pool_name}) - Stage 6 Crops (Green=Evidence, Red=Filtered) | {yield_str}", fontsize=11, fontweight='bold')
        plt.tight_layout()
        plt.show()
        plt.close(fig_ext)
    else:
        print(f"  [Notice] No Stage 6 micro-crops evaluated for {sample_id}_{pool_name}.")


print('=' * 75)
print(f'STARTING FAIL-SAFE INCREMENTAL BENCHMARK LOOP: PILOT 10 (Pool Mode: {POOL_MODE})')
print('=' * 75)

# 1. Ingest Pilot 10 Annotation Data
with open(PILOT_DATASET_PATH, 'r', encoding='utf-8') as f:
    pilot_samples = json.load(f)

print(f'[Dataset] Ingested {len(pilot_samples)} samples from {PILOT_DATASET_PATH}')

# 2. Scan Existing Checkpoints for Resume Invariant
completed_keys = set()
if os.path.exists(PREDICTIONS_JSONL):
    with open(PREDICTIONS_JSONL, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                rec = json.loads(line)
                ckey = f"{rec.get('sample_id')}_{rec.get('pool_name')}"
                completed_keys.add(ckey)
            except Exception:
                pass
    print(f'[Resume] Found {len(completed_keys)} existing predictions in {PREDICTIONS_JSONL}.')

# Slicing samples
target_samples = pilot_samples[:1] if SMOKE_TEST_MODE else pilot_samples
if MAX_SAMPLES:
    target_samples = target_samples[:MAX_SAMPLES]

# Build evaluation tasks based on POOL_MODE
eval_tasks = []
for s in target_samples:
    sid = s.get('sample_id')
    pools = s.get('pools', {})
    if POOL_MODE == 'N_5':
        if 'N_5' in pools:
            eval_tasks.append((s, 'N_5', pools['N_5']))
    else:  # 'ALL'
        for pname in ['N_5', 'N_10', 'N_20']:
            if pname in pools:
                eval_tasks.append((s, pname, pools[pname]))

print(f'[Execution Plan] Total Tasks to Process: {len(eval_tasks)} (Mode: {POOL_MODE})')

# 3. Execute Loop with Append-Only Physical Disk Flushing
session_processed = 0
total_latencies = []

# Build fast lookup for ground-truth metadata
gt_lookup = {}
for s in pilot_samples:
    for g_img in s.get('ground_truth', []):
        gt_lookup[g_img['image_id']] = g_img

with open(PREDICTIONS_JSONL, 'a', encoding='utf-8') as out_f:
    for task_idx, (sample, pool_name, pool_data) in enumerate(eval_tasks):
        sample_id = sample.get('sample_id', f'sample_{task_idx}')
        task_key = f'{sample_id}_{pool_name}'
        
        if task_key in completed_keys:
            continue
            
        query_info = sample.get('query', {})
        question = query_info.get('question', '')
        answer_info = sample.get('answer', {})
        gt_list = answer_info.get('list', [])
        gt_split = answer_info.get('split_comma', '')
        gt_freeform = answer_info.get('freeform', '')
        
        # Resolve Candidate Images for this Pool
        pool_images_meta = pool_data.get('images', [])
        candidate_images = []
        candidate_meta = []
        
        print(f'\n[{task_idx + 1}/{len(eval_tasks)}] Preparing Task: {task_key} ({len(pool_images_meta)} candidate images)...')
        for p_img in pool_images_meta:
            iid = p_img.get('image_id')
            role = p_img.get('role', 'unknown')
            gt_meta = gt_lookup.get(iid, {})
            url = gt_meta.get('original_url')
            local_p = gt_meta.get('local_path')
            
            img_pil = resolve_or_download_image(image_id=iid, url=url, rel_path=local_p)
            candidate_images.append(img_pil)
            candidate_meta.append({
                'image_id': iid,
                'role': role,
                'size': list(img_pil.size)
            })
            
        # Deepcopy config to preserve pristine base budgets before N-scaling in search()
        task_config = copy.deepcopy(config_pilot)
        n_cand = len(candidate_images)
        print(f'  [Budget Scaling] N={n_cand} images -> SAM3 cap: {task_config.max_sam3_eval_patches * n_cand}, Latent cap: {task_config.max_latent_evals * n_cand}')
        
        # Execute MSSearch Multi-Source Search
        t0 = time.perf_counter()
        try:
            response = get_mssearch_response(
                sam_model=sam3_engine,
                zoom_model=qwen_engine,
                siglip_model=siglip_engine,
                nlp_model=nlp_parser,
                annotation={'question': question, 'sample_id': sample_id, 'answer_type': 'attribute_list'},
                candidate_images=candidate_images,
                config=task_config
            )
        except Exception as e:
            print(f'[Task {task_key}] Runtime exception in get_mssearch_response: {e}')
            response = {
                'primary_answer': f'[ERROR: {str(e)}]',
                'answers': [],
                'telemetry': {'route': 'error_exception', 'timings_sec': {}},
                'extractions': []
            }
            
        latency = round(time.perf_counter() - t0, 3)
        total_latencies.append(latency)
        session_processed += 1
        
        pred_answer = response.get('primary_answer', '')
        pred_list = response.get('prediction_list', response.get('answers', []))
        telemetry = response.get('telemetry', {})
        route = telemetry.get('route', 'unknown')
        discovered_evidence = response.get('evidence_images', [])
        extractions = response.get('extractions', [])
        stage6_boxes = response.get('stage6_patch_boxes', [])
        
        # Serialize Record
        record = {
            'sample_id': sample_id,
            'id': sample_id,
            'pool_name': pool_name,
            'pool_N': pool_data.get('N', len(candidate_images)),
            'query': query_info,
            'question': question,
            'ground_truth_list': gt_list,
            'ground_truth_split': gt_split,
            'ground_truth_freeform': gt_freeform,
            'candidate_images': candidate_meta,
            'mssearch_prediction': pred_answer,
            'mssearch_prediction_list': pred_list,
            'raw_vlm_response': response.get('raw_vlm_response', ''),
            'extracted_answers': response.get('answers', []),
            'extractions': extractions,
            'stage6_patch_boxes': stage6_boxes,
            'discovered_evidence_images': discovered_evidence,
            'evidence_images': discovered_evidence,
            'route': route,
            'latency_sec': latency,
            'timings_sec': telemetry.get('timings_sec', {}),
            'mssearch_output': {
                'primary_answer': pred_answer,
                'route': route,
                'telemetry': {
                    'timings_sec': telemetry.get('timings_sec', {}),
                    'total_latency_sec': latency
                }
            }
        }
        
        out_f.write(json.dumps(record, ensure_ascii=False) + '\n')
        out_f.flush()
        os.fsync(out_f.fileno())
        completed_keys.add(task_key)
        
        avg_lat = sum(total_latencies) / len(total_latencies)
        print(f"  -> Pred: '{pred_answer}' | List: {pred_list} | Route: {route} | Latency: {latency:.2f}s (Avg: {avg_lat:.2f}s)")
        
        # Render visual display for candidate original images (with gold object boxes) and stage 6 crops + answers
        visualize_sample_results(
            sample_id=sample_id,
            pool_name=pool_name,
            question=question,
            pred_answer=pred_answer,
            pred_list=pred_list,
            gt_list=gt_list,
            candidate_images=candidate_images,
            candidate_meta=candidate_meta,
            extractions=extractions,
            stage6_boxes=stage6_boxes,
            evidence_images=discovered_evidence,
            gt_lookup=gt_lookup,
            stage6_crops=telemetry.get('stage6_crops', [])
        )
        
        gc.collect()
        torch.cuda.empty_cache()

print('=' * 75)
print(f'BENCHMARK EXECUTION FINISHED! Processed: {session_processed} tasks.')
print(f'Output saved to: {PREDICTIONS_JSONL}')
print('=' * 75)


In [ ]:
import re
import pandas as pd

print('=' * 75)
print(f'INLINE ATTRIBUTE EVALUATION & SET-BASED SCORING: {PREDICTIONS_JSONL}')
print('=' * 75)

def evaluate_freeform_prediction(pred_text: str, gt_list: list):
    """
    Computes exact-set and multi-label recall/precision against ground-truth attribute list.
    """
    pred_clean = str(pred_text).lower().replace('\n', ' ').strip()
    gt_set = set(item.lower().strip() for item in gt_list)
    
    matched_gt = set()
    for gt_item in gt_set:
        pattern = r'\b' + re.escape(gt_item) + r'\b'
        if re.search(pattern, pred_clean):
            matched_gt.add(gt_item)
            
    recall = len(matched_gt) / len(gt_set) if gt_set else 0.0
    exact_match = (matched_gt == gt_set) and len(gt_set) > 0
    
    return {
        'gt_items': ', '.join(sorted(list(gt_set))),
        'matched_items': ', '.join(sorted(list(matched_gt))),
        'recall': recall,
        'exact_match': exact_match,
        'num_gt': len(gt_set),
        'num_matched': len(matched_gt)
    }

eval_records = []
if os.path.exists(PREDICTIONS_JSONL):
    with open(PREDICTIONS_JSONL, 'r', encoding='utf-8') as f:
        for line in f:
            if not line.strip():
                continue
            rec = json.loads(line)
            metrics = evaluate_freeform_prediction(
                pred_text=rec.get('mssearch_prediction', ''),
                gt_list=rec.get('ground_truth_list', [])
            )
            eval_records.append({
                'Sample ID': rec.get('sample_id'),
                'Pool': rec.get('pool_name'),
                'Question': rec.get('question'),
                'Ground Truth': metrics['gt_items'],
                'Prediction': rec.get('mssearch_prediction'),
                'Matched': metrics['matched_items'],
                'Recall (%)': f"{metrics['recall'] * 100:.1f}%",
                'Exact Match': metrics['exact_match'],
                'Route': rec.get('route'),
                'Latency (s)': rec.get('latency_sec', 0.0)
            })

if eval_records:
    df_eval = pd.DataFrame(eval_records)
    print(df_eval[['Sample ID', 'Pool', 'Ground Truth', 'Prediction', 'Recall (%)', 'Exact Match']].to_string(index=False))
    
    mean_recall = df_eval['Recall (%)'].str.rstrip('%').astype(float).mean()
    exact_acc = df_eval['Exact Match'].mean() * 100.0
    print('-' * 75)
    print(f'Overall Set Recall   : {mean_recall:.2f}%')
    print(f'Overall Exact Match  : {exact_acc:.2f}%')
    print('=' * 75)
else:
    print('No records found to evaluate.')

In [ ]:
print('=' * 75)
print('POST-FLIGHT TELEMETRY & ZERO-OOM MEMORY AUDIT')
print('=' * 75)

oom_safe = True
for i in range(torch.cuda.device_count()):
    max_alloc_gb = torch.cuda.max_memory_allocated(i) / (1024 ** 3)
    total_gb = torch.cuda.get_device_properties(i).total_memory / (1024 ** 3)
    headroom = total_gb - max_alloc_gb
    print(f'[GPU {i}] Peak VRAM: {max_alloc_gb:.2f} GB / {total_gb:.2f} GB | Safety Margin: {headroom:.2f} GB')
    if max_alloc_gb > 15.0:
        oom_safe = False

print(f'\nZero-OOM Safety Condition Maintained: {oom_safe}')
print('=' * 75)